# NeuroLink V4 — Kaggle Starter Notebook

**Dataset:** 50,317 typing sessions · 492 days · 15 behavioral features  
**Challenge:** Build a better anomaly detector than NeuroLink V4  
**Baseline to beat:** 89% detection on Dec 2024 anomaly period, WPM gap −45.9 (p<0.001)  
**Project:** [github.com/youssefzuaiter/NeuroLink](https://github.com/youssefzuaiter/NeuroLink)

---

## What this notebook does
1. Loads and explores the dataset
2. Reproduces the NeuroLink V4 baseline (Isolation Forest, OC-SVM, SPC)
3. Shows the evaluation protocol so your submission is comparable
4. Leaves space for you to plug in your own model

---

## The Three Streams

| Stream | Features |
|--------|----------|
| **Speed** | `EV`, `RSV`, `wpm_vs_50roll`, `improvement_rate`, `warmup_effect` |
| **Accuracy** | `error_entropy`, `speed_var_wl`, `worst_char_improvement`, `miss_ratio_top3`, `error_rate_trend` |
| **Motor** | `CTA`, `CTA_cos`, `hand_asymmetry`, `time_of_day`, `speed_dispersion` |

The `wpm` column is the raw WPM for that session — use it for evaluation, not as a model input.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

## 1. Load the Data

In [ ]:
# If running on Kaggle, path is /kaggle/input/neurolink-.../neurolink_features_kaggle.csv
# If running locally, adjust path accordingly
try:
    import os
    kaggle_path = '/kaggle/input/'
    dataset_dir = [d for d in os.listdir(kaggle_path) if 'neurolink' in d.lower()][0]
    DATA_PATH = f'{kaggle_path}{dataset_dir}/neurolink_features_kaggle.csv'
except:
    DATA_PATH = 'neurolink_features_kaggle.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'Sessions: {len(df):,}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
df.describe().round(3)

## 2. Aggregate to Daily Level

Models operate on daily aggregates. Each day's features are the mean of all sessions that day.

In [ ]:
FEATURE_COLS = [
    'EV', 'RSV', 'wpm_vs_50roll', 'improvement_rate', 'warmup_effect',
    'error_entropy', 'speed_var_wl', 'worst_char_improvement', 'miss_ratio_top3', 'error_rate_trend',
    'CTA', 'CTA_cos', 'hand_asymmetry', 'time_of_day', 'speed_dispersion'
]

daily = df.groupby('date').agg(
    **{f: (f, 'mean') for f in FEATURE_COLS},
    wpm=('wpm', 'mean'),
    n_sessions=('wpm', 'count')
).reset_index()

daily = daily.sort_values('date').reset_index(drop=True)
print(f'Days: {len(daily)}')
print(f'Avg sessions/day: {daily["n_sessions"].mean():.1f}')
daily.head()

## 3. Define the Evaluation Protocol

Three metrics — all submissions should report all three to be comparable:

1. **Anomaly rate** — what % of days are flagged (selectivity check — should be ~8-10%)
2. **WPM gap** — mean WPM difference between flagged vs normal days (Welch t-test)
3. **Dec 2024 detection rate** — the ground-truth anomaly period. WPM dropped to ~60 during this period. What % of those days does your model flag?

In [ ]:
# Chronological 75/25 split — same as NeuroLink V4
split_idx = int(len(daily) * 0.75)
train_daily = daily.iloc[:split_idx].copy()
test_daily  = daily.iloc[split_idx:].copy()
print(f'Train: {len(train_daily)} days ({train_daily["date"].min()} → {train_daily["date"].max()})')
print(f'Test:  {len(test_daily)} days ({test_daily["date"].min()} → {test_daily["date"].max()})')

# Ground truth: Dec 2024 anomaly period
dec24_mask = (daily['date'].dt.year == 2024) & (daily['date'].dt.month == 12)
print(f'\nDec 2024 days in dataset: {dec24_mask.sum()}')
print(f'Dec 2024 avg WPM: {daily.loc[dec24_mask, "wpm"].mean():.1f} (baseline avg: {daily["wpm"].mean():.1f})')

In [ ]:
def evaluate(flags, daily_df, label='Model'):
    """Evaluate anomaly flags against all three metrics."""
    daily_df = daily_df.copy()
    daily_df['flag'] = flags

    anomaly_rate = flags.mean() * 100

    flagged_wpm = daily_df.loc[daily_df['flag'], 'wpm']
    normal_wpm  = daily_df.loc[~daily_df['flag'], 'wpm']
    wpm_gap = normal_wpm.mean() - flagged_wpm.mean()
    _, p_val = stats.ttest_ind(normal_wpm, flagged_wpm, equal_var=False)
    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))

    dec24 = (daily_df['date'].dt.year == 2024) & (daily_df['date'].dt.month == 12)
    dec24_detection = daily_df.loc[dec24, 'flag'].mean() * 100 if dec24.sum() > 0 else float('nan')

    print(f'  {label}')
    print(f'    Anomaly rate:       {anomaly_rate:.1f}%')
    print(f'    WPM gap:            +{wpm_gap:.1f} WPM  ({sig})')
    print(f'    Dec 2024 detection: {dec24_detection:.0f}%')
    print()
    return {'anomaly_rate': anomaly_rate, 'wpm_gap': wpm_gap, 'p': p_val, 'dec24': dec24_detection}

print('Evaluation function ready.')

## 4. Baseline Methods

These are the four baselines NeuroLink V4 was compared against. Reproduce them here to confirm your environment gives the same results.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_daily[FEATURE_COLS])
X_all   = scaler.transform(daily[FEATURE_COLS])

print('Baselines on full dataset (train fit, all days evaluated):')
print('=' * 55)

# SPC — rolling mean ±2σ on WPM
roll_mean = daily['wpm'].rolling(30, min_periods=5).mean()
roll_std  = daily['wpm'].rolling(30, min_periods=5).std()
spc_flags = (daily['wpm'] < roll_mean - 2*roll_std) | (daily['wpm'] > roll_mean + 2*roll_std)
spc_flags = spc_flags.fillna(False)
evaluate(spc_flags, daily, 'SPC (rolling ±2σ)')

# Isolation Forest
iso = IsolationForest(contamination=0.10, random_state=42, n_estimators=200)
iso.fit(X_train)
iso_flags = iso.predict(X_all) == -1
evaluate(pd.Series(iso_flags), daily, 'Isolation Forest')

# One-Class SVM
ocsvm = OneClassSVM(nu=0.10, kernel='rbf', gamma='scale')
ocsvm.fit(X_train)
ocsvm_flags = ocsvm.predict(X_all) == -1
evaluate(pd.Series(ocsvm_flags), daily, 'One-Class SVM')

## 5. Visualise the Data

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle('NeuroLink Dataset — Daily Behavioral Overview', fontsize=14, fontweight='bold')

# WPM timeline
ax = axes[0]
ax.plot(daily['date'], daily['wpm'], color='#2563EB', lw=0.8, alpha=0.6)
ax.plot(daily['date'], daily['wpm'].rolling(30).mean(), color='#2563EB', lw=2, label='30-day avg')
dec24 = (daily['date'].dt.year==2024) & (daily['date'].dt.month==12)
ax.axvspan(daily.loc[dec24,'date'].min(), daily.loc[dec24,'date'].max(),
           alpha=0.15, color='red', label='Dec 2024 anomaly period')
ax.set_ylabel('WPM'); ax.legend(fontsize=8)
ax.set_title('Speed — Words Per Minute')

# Error entropy
ax = axes[1]
ax.plot(daily['date'], daily['error_entropy'], color='#D97706', lw=0.8, alpha=0.6)
ax.plot(daily['date'], daily['error_entropy'].rolling(30).mean(), color='#D97706', lw=2)
ax.axvspan(daily.loc[dec24,'date'].min(), daily.loc[dec24,'date'].max(), alpha=0.15, color='red')
ax.set_ylabel('Entropy'); ax.set_title('Accuracy — Error Entropy')

# Hand asymmetry
ax = axes[2]
ax.plot(daily['date'], daily['hand_asymmetry'], color='#059669', lw=0.8, alpha=0.6)
ax.plot(daily['date'], daily['hand_asymmetry'].rolling(30).mean(), color='#059669', lw=2)
ax.axvspan(daily.loc[dec24,'date'].min(), daily.loc[dec24,'date'].max(), alpha=0.15, color='red')
ax.set_ylabel('Asymmetry'); ax.set_title('Motor — Hand Asymmetry')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()
print('Red shading = Dec 2024 ground-truth anomaly period.')

## 6. Your Model Goes Here

Plug in your approach below. The only requirement is that you produce a boolean Series `your_flags` aligned with `daily` and pass it to `evaluate()`.

**Ideas to try:**
- LSTM / GRU autoencoder on the 15-feature time series
- Transformer-based anomaly detection (PatchTST, Anomaly Transformer)
- Differentially private autoencoder with tighter ε via public pre-training
- Stream-specific models — separate detector per stream, combine signals
- Local outlier factor on sliding windows

**The challenge:** NeuroLink V4 hits 89% on Dec 2024 with formal DP (ε=10). Can you get higher — or achieve the same with stronger privacy?

In [ ]:
# ── YOUR MODEL HERE ──────────────────────────────────────────────────────────

# Example skeleton:
# from your_library import YourModel
# model = YourModel(...)
# model.fit(X_train)
# your_flags = pd.Series(model.predict(X_all) == -1)
# evaluate(your_flags, daily, 'Your Model Name')

print('Ready for your model.')

## 7. Submit Your Results

When you have results, share them in the dataset comments with:
- Method name and brief description
- Anomaly rate, WPM gap, Dec 2024 detection %
- Link to your notebook

The best approaches will be cited in the NeuroLink V5 paper.

---

**Citation:**  
Zuaiter, Y. (2026). NeuroLink: A Privacy-Preserving Multi-Stream Behavioral Baseline Engine for Personal Typing Performance Monitoring. GitHub: https://github.com/youssefzuaiter/NeuroLink